# Joung 2023 — Data Preprocessing

Prepares the TF Atlas (Joung et al. 2023) for downstream GRN inference and modelling.

**Pipeline overview**

| Step | Description |
|------|-------------|
| 0 | Load raw TF Atlas (`TFAtlas_raw`) and auxiliary datasets |
| 1 | Annotate cells: parse TF identity, flag differentiated and combinatorially-tested cells |
| 2 | Filter TFs with fewer than `min_num_cells_per_tf` cells; subsample to `max_num_cells_per_tf` |
| 3 | Sample control cells (mCherry / GFP); concatenate with perturbed cells |
| 4 | Normalize (CPM) → log1p; select HVGs; compute PCA + UMAP |
| 5 | Inspect control overlap on UMAP; merge mCherry and GFP into a single `ctrl` label |
| 6 | Save final object |

**Input files** (place under `data/real/Joung2023/`)
- `GSE217460_210322_TFAtlas_raw.h5ad` — main single-cell atlas
- `GSE217460_210322_TFAtlas_differentiated_raw.h5ad` — differentiated-cell subset
- `GSE217066_210715_combinatorial_subsample.h5ad` — combinatorial perturbation subset

**Output files**
- `Joung2023_before_preprocessing.h5ad` — filtered/subsampled, raw counts only
- `Joung2023.h5ad` — final object with `counts` and `lognorm` layers, PCA, UMAP

## 0. Imports and parameters

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
# --- Input / output paths ---
data_folder                = '../../../data/real/Joung2023/'
adata_file                 = data_folder + 'GSE217460_210322_TFAtlas_raw.h5ad'
adata_differentiated_file  = data_folder + 'GSE217460_210322_TFAtlas_differentiated_raw.h5ad'
adata_combinatorial_file   = data_folder + 'GSE217066_210715_combinatorial_subsample.h5ad'

# --- Filtering parameters ---
min_num_cells_per_tf = 500   # TFs with fewer cells are excluded
max_num_cells_per_tf = 1000  # TFs with more cells are randomly subsampled
num_ctrl_cells       = 10000 # total control cells sampled (mCherry + GFP combined)
seed                 = 42

## 1. Load data and annotate cells

In [ ]:
adata = sc.read_h5ad(adata_file)
adata

In [ ]:
# The TF column encodes both ORF ID and gene name (e.g. "TFORF0867-NR1H2"); split them.
adata.obs[['ORF_ID', 'TF_name']] = adata.obs['TF'].str.split('-', n=1, expand=True)

In [ ]:
adata_differentiated  = sc.read_h5ad(adata_differentiated_file)
adata_combinatorial   = sc.read_h5ad(adata_combinatorial_file)

In [ ]:
# Flag cells that appear in the differentiated subset (used as a covariate later).
adata.obs['is_differentiated'] = adata.obs_names.isin(adata_differentiated.obs_names)

In [ ]:
# Restrict to double perturbations (exactly one comma) and extract the two TF names.
adata_combinatorial = adata_combinatorial[adata_combinatorial.obs['TF'].str.count(',') == 1, :]
adata_combinatorial.obs[['TF1', 'TF2']]         = adata_combinatorial.obs['TF'].str.split(',', n=1, expand=True)
adata_combinatorial.obs[['ORF_ID1', 'TF_name1']] = adata_combinatorial.obs['TF1'].str.split('-', n=1, expand=True)
adata_combinatorial.obs[['ORF_ID2', 'TF_name2']] = adata_combinatorial.obs['TF2'].str.split('-', n=1, expand=True)

double_intervened_tfs = list(
    set(adata_combinatorial.obs['TF_name1'].unique()) |
    set(adata_combinatorial.obs['TF_name2'].unique())
)

In [ ]:
# Flag TFs that also appear in combinatorial perturbation experiments.
adata.obs['is_combinatorially_tested'] = adata.obs['TF_name'].isin(double_intervened_tfs)
print(f"Cells with combinatorially-tested TF: {adata.obs['is_combinatorially_tested'].sum():,}")

## 2. Quality control

The raw atlas has already passed QC (all cells have ≥700 genes and ≤5 % mitochondrial reads). We verify this and proceed without further filtering.

In [ ]:
adata.obs[['n_counts', 'n_genes', 'percent_mito']].agg(['min', 'max'], numeric_only=True)

## 3. Cell subsampling

- Drop TFs represented by fewer than `min_num_cells_per_tf` cells.
- Subsample TFs with more than `max_num_cells_per_tf` cells to cap class imbalance.
- Sample `num_ctrl_cells` control cells (mCherry + GFP pooled) separately.

In [ ]:
tf_name_counts = adata.obs['TF_name'].value_counts()
to_remove      = tf_name_counts[tf_name_counts < min_num_cells_per_tf].index
adata          = adata[~adata.obs['TF_name'].isin(to_remove), :]
print(f"TFs retained: {adata.obs['TF_name'].nunique()}  |  Cells: {adata.n_obs:,}")

In [ ]:
# Inspect differentiation fraction per TF — informational, not used for filtering.
diff_stats = adata.obs.groupby('TF_name')['is_differentiated'].agg(['sum', 'count'])
diff_stats.columns = ['n_differentiated', 'n_total']
diff_stats['n_non_differentiated'] = diff_stats['n_total'] - diff_stats['n_differentiated']
diff_stats['frac_differentiated']  = diff_stats['n_differentiated'] / diff_stats['n_total']

print(f"TFs with ALL cells differentiated:     {(diff_stats['n_non_differentiated'] == 0).sum()}")
print(f"TFs with NO  cells differentiated:     {(diff_stats['n_differentiated']     == 0).sum()}")
print(f"TFs with MIXED differentiation status: {((diff_stats['n_differentiated'] > 0) & (diff_stats['n_non_differentiated'] > 0)).sum()}")
print()
print(diff_stats.sort_values('frac_differentiated', ascending=False))
print('\nFrac differentiated summary:')
print(diff_stats['frac_differentiated'].describe())

In [ ]:
def subsample_tf(group, max_cells, seed=42):
    """Return up to max_cells rows from group, sampled without replacement."""
    if len(group) <= max_cells:
        return group
    return group.sample(n=max_cells, random_state=seed)

keep_idx = (
    adata.obs
    .groupby('TF_name', group_keys=False)
    .apply(lambda g: subsample_tf(g, max_num_cells_per_tf), include_groups=False)
    .index
)

# Exclude control labels from the perturbed set; controls are sampled separately below.
intervention_adata = adata[keep_idx, :]
intervention_adata = intervention_adata[~intervention_adata.obs['TF_name'].isin(['mCherry', 'GFP']), :]
print(intervention_adata.obs['TF_name'].value_counts().describe())

In [ ]:
# Sample control cells from the full (pre-subsampling) pool to avoid
# the per-TF cap artificially reducing control representation.
ctrl_mask      = adata.obs['TF_name'].isin(['mCherry', 'GFP'])
selected_idx   = adata.obs[ctrl_mask].sample(n=num_ctrl_cells, random_state=seed).index
ctrl_adata     = adata[selected_idx, :]
print(f"Control cells: {ctrl_adata.n_obs:,}")

In [ ]:
adata = ad.concat([intervention_adata, ctrl_adata], axis=0)
print(f"Combined dataset: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## 4. Normalization, HVG selection, PCA, and UMAP

In [ ]:
# Save the filtered object with raw counts before any normalization.
adata.write_h5ad(data_folder + 'Joung2023_before_preprocessing.h5ad')
print(f"Saved → {data_folder}Joung2023_before_preprocessing.h5ad")

In [ ]:
# Preserve raw counts and lognorm as layers so downstream notebooks
# can choose the appropriate representation without re-running this notebook.
adata.layers['counts'] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers['lognorm'] = adata.X.copy()

# HVG selection is done on lognorm data, stratified by batch.
sc.pp.highly_variable_genes(adata, n_top_genes=5000, batch_key='batch', subset=False)
print(f"HVGs selected: {adata.var['highly_variable'].sum()}")
adata = adata[:, adata.var['highly_variable']]

# Scale only for PCA; X is restored to lognorm afterwards.
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver='arpack')
adata.X = adata.layers['lognorm'].copy()

sc.pp.neighbors(adata, n_pcs=50)
sc.tl.umap(adata)

adata

## 5. Control inspection: mCherry vs GFP on UMAP

The two fluorescent controls should occupy the same transcriptional space. If they do, it is safe to merge them into a single `ctrl` label.

In [ ]:
ctrl_colors = {'mCherry': '#B8860B', 'GFP': '#006400'}
adata_ctrl  = adata[adata.obs['TF_name'].isin(ctrl_colors)]
print(adata_ctrl.obs['TF_name'].value_counts())

fig, ax = plt.subplots(figsize=(6, 5))
for tf, grp in adata_ctrl.obs.groupby('TF_name'):
    coords = adata_ctrl[grp.index].obsm['X_umap']
    ax.scatter(coords[:, 0], coords[:, 1], c=ctrl_colors[tf], label=tf, s=3, alpha=0.5)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP — mCherry vs GFP controls')
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay 20 perturbed TFs spanning the full differentiation-fraction spectrum
# alongside the controls, to verify that controls sit in the unperturbed region.
diff_frac    = adata.obs.groupby('TF_name')['is_differentiated'].mean()
diff_frac    = diff_frac.drop(labels=['mCherry', 'GFP'], errors='ignore')
selected_tfs = (diff_frac
                .quantile(np.linspace(0, 1, 20), interpolation='nearest')
                .drop_duplicates()
                .values.tolist())
selected_tfs = diff_frac[diff_frac.isin(selected_tfs)].index.tolist()
plot_tfs     = ['mCherry', 'GFP'] + selected_tfs
print(f"Plotting {len(plot_tfs)} groups: 2 controls + {len(selected_tfs)} TFs")

adata_plot = adata[adata.obs['TF_name'].isin(plot_tfs)]
cmap       = cm.get_cmap('coolwarm', len(selected_tfs))
tf_colors  = {tf: cmap(i) for i, tf in enumerate(sorted(selected_tfs, key=lambda t: diff_frac[t]))}
tf_colors.update({'mCherry': '#B8860B', 'GFP': '#006400'})

fig, ax = plt.subplots(figsize=(8, 6))
for tf, grp in adata_plot.obs.groupby('TF_name'):
    coords     = adata_plot[grp.index].obsm['X_umap']
    is_control = tf in ('mCherry', 'GFP')
    frac       = diff_frac.get(tf, float('nan'))
    label      = tf if is_control else f"{tf} ({frac:.2f})"
    ax.scatter(coords[:, 0], coords[:, 1],
               c=[tf_colors[tf]], label=label,
               s=4 if is_control else 2,
               alpha=0.8 if is_control else 0.4,
               zorder=3 if is_control else 1)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP — controls + 20 TFs spanning differentiation spectrum\n(label: TF name, differentiation fraction)')
ax.legend(markerscale=4, fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# mCherry and GFP overlap on UMAP → merge into a single 'ctrl' label.
# The original fluorescent identity is preserved in 'control_type' for traceability.
adata.obs['control_type'] = adata.obs['TF_name'].where(
    adata.obs['TF_name'].isin(['mCherry', 'GFP']), other=None
)
adata.obs['TF_name'] = adata.obs['TF_name'].cat.rename_categories(
    {c: 'ctrl' for c in ['mCherry', 'GFP'] if c in adata.obs['TF_name'].cat.categories}
)

print(adata.obs['control_type'].value_counts(dropna=False))
print(adata.obs['TF_name'].value_counts().head())

## 6. Save

In [ ]:
out_path = data_folder + 'Joung2023.h5ad'
adata.write_h5ad(out_path)
print(f"Saved → {out_path}")
print(adata)